# Montando o Drive e conectando ao modelo

In [25]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [26]:
import os

caminho_modelo = "/content/drive/MyDrive/techchallenge_fase3/biomistral-medquad-lora"

if os.path.exists(caminho_modelo):
    arquivos = os.listdir(caminho_modelo)
    print(f"✅ Modelo encontrado! {len(arquivos)} arquivos:")
    for arq in arquivos:
        print(f"   - {arq}")
else:
    print(" Modelo NÃO encontrado em:", caminho_modelo)


✅ Modelo encontrado! 7 arquivos:
   - tokenizer.model
   - README.md
   - adapter_model.safetensors
   - adapter_config.json
   - chat_template.jinja
   - tokenizer_config.json
   - tokenizer.json


## Instalar dependências

In [27]:
!pip install -q "datasets<3.0"

!python scripts/setup_data_colab.py

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 6.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
trl 0.24.0 requires datasets>=3.0.0, but you have datasets 2.21.0 which is incompatible.
unsloth-zoo 2026.9.2 requires datasets!=4.0.*,!=4.1.0,<4.4.0,>=3.4.1, but you have datasets 2.21.0 which is incompatible.
[19:23:16] ======================================================================
[19:23:16] 📚 [1/4] MedQuAD (NIH)
[19:23:16] ======================================================================
[19:23:16] 📥 Baixando MedQuAD do GitHub oficial...
[19:23:16] ======================================================================
[19:23:16] 📚 [2/4] ChatBulário (HuggingFace)
[19:23:16] ======================================================================
[19:23:17] 📥 Baixando ChatBulário do HuggingFace...
Generating train split: 1

In [31]:
# Força uninstall da versão 2.x e instala a 1.x
!pip uninstall -y numpy
!pip install -q "numpy==1.26.4"

# Verifica
import numpy as np
print(f"numpy {np.__version__}")
print(f"np.float_ disponível: {hasattr(np, 'float_')}")

Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 76.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
trl 0.24.0 requires datasets>=3.0.0, but you have datasets 2.21.0 which is incompatible.
unsloth-zoo 2026.9.2 requires datasets!=4.0.*,!=4.1.0,<4.4.0,>=3.4.1, but you have datasets 2.21.0 which is incompatible.
shap 0.52.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 5.0.0.93 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
tifffile 2026.8.23 requires numpy>=2.1, b

In [32]:
# 1. Desinstala TODOS os pydantic
!pip uninstall -y pydantic pydantic-settings pydantic-core 2>&1 | tail -3

# 2. Limpa cache
!pip cache purge 2>&1 | tail -2

# 3. Instala pydantic v2 (compatível com gradio e ambiente atual)
!pip install -q "pydantic>=2.0,<3.0" "pydantic-settings"

# 4. Verifica TODOS os locais onde pydantic pode estar
import subprocess
result = subprocess.run(
    ["find", "/usr/local/lib", "/usr/lib", "-name", "pydantic", "-type", "d"],
    capture_output=True, text=True
)
print("📂 Localizações do pydantic:")
print(result.stdout)

# 5. Testa
import pydantic
import pydantic_core
print(f"\n✅ pydantic {pydantic.__version__}")
print(f"✅ pydantic_core {pydantic_core.__version__}")
print(f"✅ AliasChoices disponível: {hasattr(pydantic, 'AliasChoices') or hasattr(pydantic_core, 'AliasChoices')}")

# 6. Testa unsloth
# O erro de incompatibilidade do NumPy ocorre porque a versão antiga
# continua na memória. Reinicie a sessão do Colab para resolver.
try:
    from unsloth import FastLanguageModel
    print("✅ Unsloth importado com sucesso!")
except ImportError as e:
    print(f"❌ Unsloth falhou: {e}")
except ValueError as e:
    print(f"❌ Erro de versão do NumPy detectado: {e}\n⚠️ AÇÃO NECESSÁRIA: Vá em 'Ambiente de execução' -> 'Reiniciar sessão' e rode novamente.")


Found existing installation: pydantic_core 2.46.5
Uninstalling pydantic_core-2.46.5:
  Successfully uninstalled pydantic_core-2.46.5
Files removed: 65
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.2/110.2 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.6/472.6 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 8.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langgraph-sdk 0.4.3 requires websockets<17,>=14, but you have websockets 12.0 which is incompatible.
google-genai 2.12.1 requires websockets<17.0,>=13.0.0, but you have websockets 12.0 which is incompatible.
langsmith 0.11.1 requires websockets>=15.0, but you have websockets 12.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>

/usr/local/lib/python3.13/dist-packages/unsloth/__init__.py:1543: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


In [33]:

!pip install -q "chromadb==0.4.18"
!pip install -q "gradio==4.44.0"
!pip install -q "huggingface_hub" "transformers"

!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q "bitsandbytes" "accelerate" "trl" "peft"

!pip install -q "reportlab"


import torch
print(f"✅ PyTorch {torch.__version__}")
print(f"✅ CUDA disponível: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.6/224.6 kB 5.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 1.3.0 requires websockets<13.0,>=10.0, but you have websockets 17.1 which is incompatible.
hf-gradio 0.4.1 requires gradio-client<3.0,>=2.0, but you have gradio-client 1.3.0 which is incompatible.
langgraph-sdk 0.4.3 requires websockets<17,>=14, but you have websockets 17.1 which is incompatible.
google-genai 2.12.1 requires websockets<17.0,>=13.0.0, but you have websockets 17.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.44.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.44.0 which is incompatible.
google-adk 2.7.1 requires websockets<16,>=15.0.1, but you have websockets 17.1 which

## Clonar repositório

## Passo 5

In [34]:
import os
from pathlib import Path

# Limpa qualquer clone anterior
if Path("Techchalleng3").exists():
    !rm -rf Techchalleng3

# Clona o repo
!git clone https://github.com/Flamers-Team/Techchalleng3.git

# Entra na pasta
%cd Techchalleng3

# Confirma
print(f"📁 Estamos em: {os.getcwd()}")
print(f"📂 Arquivos: {os.listdir('.')[:10]}")

Cloning into 'Techchalleng3'...
remote: Enumerating objects: 11632, done.
remote: Counting objects: 100% (11632/11632), done.
remote: Compressing objects: 100% (4726/4726), done.
remote: Total 11632 (delta 6965), reused 11534 (delta 6877), pack-reused 0 (from 0)
Receiving objects: 100% (11632/11632), 10.81 MiB | 23.11 MiB/s, done.
Resolving deltas: 100% (6965/6965), done.
Filtering content: 100% (9/9), 471.74 MiB | 8.25 MiB/s, done.
/content/Techchalleng3/Techchalleng3
📁 Estamos em: /content/Techchalleng3/Techchalleng3
📂 Arquivos: ['.gitignore', 'docs', '.git', 'README.md', 'src', 'notebooks', 'data', '.gitattributes', 'scripts']


## Passo 6

In [35]:
import shutil
from pathlib import Path

DRIVE_MODEL = Path("/content/drive/MyDrive/techchallenge_fase3/biomistral-medquad-lora")
LOCAL_MODEL = Path("/content/biomistral-medquad-lora")

if not LOCAL_MODEL.exists() and DRIVE_MODEL.exists():
    print(f"📦 Copiando modelo do Drive...")
    shutil.copytree(DRIVE_MODEL, LOCAL_MODEL)
    print(f"✅ Copiado! {len(list(LOCAL_MODEL.iterdir()))} arquivos")
elif LOCAL_MODEL.exists():
    print(f"✅ Modelo já está em {LOCAL_MODEL}")
else:
    print(f"❌ Drive não tem o modelo em {DRIVE_MODEL}")

✅ Modelo já está em /content/biomistral-medquad-lora


## Passo 7

In [36]:
import gradio_client
import os

gc_path = os.path.join(os.path.dirname(gradio_client.__file__), "utils.py")

with open(gc_path, "r") as f:
    content = f.read()

# Patch TODAS as ocorrências (não só 'const')
patches = [
    ('    if "enum" in schema:', '    if isinstance(schema, dict) and "enum" in schema:'),
    ('    if "const" in schema:', '    if isinstance(schema, dict) and "const" in schema:'),
    ('if "enum" in schema:', 'if isinstance(schema, dict) and "enum" in schema:'),
    ('if "const" in schema:', 'if isinstance(schema, dict) and "const" in schema:'),
]

patched = 0
for bug, fix in patches:
    if bug in content and fix not in content:
        content = content.replace(bug, fix)
        patched += 1

if patched > 0:
    with open(gc_path, "w") as f:
        f.write(content)
    print(f"✅ {patched} patches aplicados no gradio_client")
else:
    print("ℹ️  Patch já estava aplicado")


ℹ️  Patch já estava aplicado


## Passo 8

In [37]:
import os
os.chdir("/content/Techchalleng3")

with open("src/ui/gradio_app.py", "r") as f:
    code = f.read()

if "share=False" in code:
    code = code.replace("share=False", "share=True")
    with open("src/ui/gradio_app.py", "w") as f:
        f.write(code)
    print("✅ share=False → share=True")
else:
    print("ℹ️  share já estava True")

ℹ️  share já estava True


## Passo 9

In [38]:
# Copia modelo pro caminho que a UI espera (/content/Techchalleng3/biomistral-medquad-lora)
import shutil
from pathlib import Path

SOURCE = Path("/content/biomistral-medquad-lora")
TARGET = Path("/content/Techchalleng3/biomistral-medquad-lora")

if not TARGET.exists() and SOURCE.exists():
    print("📦 Copiando modelo pro caminho que a UI espera...")
    shutil.copytree(SOURCE, TARGET)
    print(f"✅ Copiado! {len(list(TARGET.iterdir()))} arquivos")
elif TARGET.exists():
    print(f"✅ Modelo já está em {TARGET}")
else:
    print(f"❌ Modelo não encontrado em {SOURCE}")


✅ Modelo já está em /content/Techchalleng3/biomistral-medquad-lora


## ⭐ PASSO EXTRA: Carregar dados do Google Drive

**Setup necessário (uma única vez):**
1. Faça upload da pasta `data/` (do seu PC) para `My Drive/techchallenge_fase3/data/` no Google Drive
2. Estrutura esperada:
```
techchallenge_fase3/
└── data/
    ├── raw/
    │   ├── chatbulario_train.jsonl
    │   ├── chatbulario_validation.jsonl
    │   ├── chatbulario_test.jsonl
    │   ├── medquad_finetuning.jsonl
    │   ├── cid10_subcategorias.csv
    │   └── synthetic_clinical_notes/
    └── processed/
        ├── train.jsonl / val.jsonl / test.jsonl
        ├── synthetic_clinical_notes_anonimizado.jsonl
        └── chroma_index/  ← já indexado! (10k chatbulario + 22 cid10 + 1k synthetic)
```

**O que essa célula faz:**
- Copia `data/raw/` e `data/processed/` do Drive para `/content/Techchalleng3/data/`
- Inclui o ChromaDB já indexado (não precisa reindexar!)
- Tempo: ~1-3 min (depende do tamanho)

**Se não tiver os dados no Drive**, use o script automático:
```bash
!python scripts/setup_data_colab.py
!python src/rag/build_index_chatbulario.py 10000
```


In [ ]:
# ============================================================
# PASSO EXTRA: Baixar dados pré-processados do Drive
# ============================================================
import os
import shutil
import zipfile
from pathlib import Path
import time

t0 = time.time()

# Caminhos do Drive (pasta techchallenge_fase3/data)
DRIVE_DATA_DIR = Path("/content/drive/MyDrive/techchallenge_fase3/data")
TARGET_DATA_DIR = Path("/content/Techchalleng3/data")

# Monta Drive (caso ainda não tenha)
from google.colab import drive
if not Path("/content/drive/MyDrive").exists():
    drive.mount('/content/drive', force_remount=False)

# Verifica se existe a pasta no Drive
if not DRIVE_DATA_DIR.exists():
    print(f"❌ Pasta não encontrada: {DRIVE_DATA_DIR}")
    print(f"   Você subiu a pasta data/ pro Google Drive?")
    print(f"   Esperado em: /content/drive/MyDrive/techchallenge_fase3/data/")
else:
    print(f"✅ Pasta encontrada: {DRIVE_DATA_DIR}")
    print(f"📂 Conteúdo:")
    for item in sorted(DRIVE_DATA_DIR.rglob('*'))[:15]:
        if item.is_file():
            size_mb = item.stat().st_size / 1024 / 1024
            rel = item.relative_to(DRIVE_DATA_DIR)
            print(f"   📄 {rel} ({size_mb:.1f} MB)")
    
    # Calcula tamanho total
    total_mb = sum(f.stat().st_size for f in DRIVE_DATA_DIR.rglob('*') if f.is_file()) / 1024 / 1024
    print(f"\n📊 Tamanho total: {total_mb:.1f} MB")
    
    # Copia pra /content/Techchalleng3/data/
    if TARGET_DATA_DIR.exists():
        print(f"\n🗑️  Removendo data/ existente...")
        shutil.rmtree(TARGET_DATA_DIR)
    TARGET_DATA_DIR.mkdir(parents=True, exist_ok=True)
    
    print(f"📦 Copiando {DRIVE_DATA_DIR} → {TARGET_DATA_DIR}...")
    
    # Copia raw/ e processed/ separadamente
    for subdir in ["raw", "processed"]:
        src = DRIVE_DATA_DIR / subdir
        dst = TARGET_DATA_DIR / subdir
        if src.exists():
            shutil.copytree(src, dst)
            sub_mb = sum(f.stat().st_size for f in dst.rglob('*') if f.is_file()) / 1024 / 1024
            print(f"   ✅ {subdir}/ copiado ({sub_mb:.1f} MB)")
    
    print(f"\n⏱️  Tempo: {time.time() - t0:.1f}s")
    print(f"\n🎉 Dados prontos em {TARGET_DATA_DIR}")
    print(f"   Pula direto pro PASSO 9 (carregar RAG + LLM)")


## ChatBulário COMPLETO

In [39]:
!python src/rag/build_index_chatbulario.py

📥 INDEXAÇÃO DO CHATBULÁRIO NO CHROMADB

📂 Carregando amostras do ChatBulário...
   Total disponível: 68,938 pares Q&A

🔌 Conectando ao ChromaDB em /content/Techchalleng3/data/processed/chroma_index...
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:299: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(
Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpyth

### CORREÇÃO DE ERRO no chromadb

In [40]:
# Substitui np.float_ por np.float64 diretamente no código do chromadb
import subprocess
result = subprocess.run(
    ["grep", "-rl", "np.float_", "/usr/local/lib/python3.13/dist-packages/chromadb/"],
    capture_output=True, text=True
)
files_to_patch = result.stdout.strip().split("\n")
print(f"Encontrado em {len(files_to_patch)} arquivos")

for f in files_to_patch:
    if f and f.endswith(".py"):
        with open(f, "r") as fp:
            content = fp.read()
        content = content.replace("np.float_", "np.float64")
        with open(f, "w") as fp:
            fp.write(content)
        print(f"✅ Patch: {f}")

# Verifica
import chromadb
print(f"✅ chromadb {chromadb.__version__}")

Encontrado em 1 arquivos
✅ chromadb 0.4.18


# RAG

In [42]:
# ============================================================
# FIX numpy/numPy incompatibilidade
# ============================================================
import subprocess

# 1. Desinstala TUDO relacionado a numpy/sentence-transformers
print("🗑️  Removendo conflitos...")
subprocess.run(["pip", "uninstall", "-y",
                "numpy", "sentence-transformers", "transformers"],
               capture_output=True)

# 2. Limpa cache
subprocess.run(["pip", "cache", "purge"], capture_output=True)

# 3. Instala numpy 1.26.4 PRIMEIRO
print("📦 Instalando numpy 1.26.4...")
subprocess.run(["pip", "install", "-q", "numpy==1.26.4"],
               capture_output=True)

# 4. DEPOIS instala sentence-transformers
print("📦 Instalando sentence-transformers...")
subprocess.run(["pip", "install", "-q",
                "sentence-transformers==2.2.2", "transformers==4.36.0"],
               capture_output=True)

# 5. Verifica
import numpy as np
print(f"✅ numpy {np.__version__}")
print(f"✅ np.float_ disponível: {hasattr(np, 'float_')}")

# 6. Testa sentence-transformers
try:
    from sentence_transformers import SentenceTransformer
    print("✅ sentence-transformers importado!")

    # Testa modelo (vai baixar ~80 MB na primeira vez)
    model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    vec = model.encode("teste")
    print(f"✅ Modelo carregado! Vetor shape: {vec.shape}")
except Exception as e:
    print(f"❌ Erro: {e}")

🗑️  Removendo conflitos...
📦 Instalando numpy 1.26.4...
📦 Instalando sentence-transformers...
✅ numpy 2.1.3
✅ np.float_ disponível: False
❌ Erro: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject


In [44]:
!pip install --force-reinstall --no-deps \
    "numpy==1.26.4" \
    "sentence-transformers==2.2.2" \
    "transformers==4.36.0"

  Using cached numpy-1.26.4-cp313-cp313-linux_x86_64.whl
  Using cached sentence_transformers-2.2.2-py3-none-any.whl
  Using cached transformers-4.36.0-py3-none-any.whl.metadata (126 kB)
Using cached transformers-4.36.0-py3-none-any.whl (8.2 MB)
  Attempting uninstall: transformers
    Found existing installation: transformers 4.36.0
    Uninstalling transformers-4.36.0:
      Successfully uninstalled transformers-4.36.0
  Attempting uninstall: sentence-transformers
    Found existing installation: sentence-transformers 2.2.2
    Uninstalling sentence-transformers-2.2.2:
      Successfully uninstalled sentence-transformers-2.2.2
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4


In [43]:
import os
import time
from pathlib import Path
os.chdir("/content/Techchalleng3")

t0 = time.time()

# 1. RAG
print("=" * 70)
print("📚 [1/2] Carregando RAG (ChromaDB)...")
print("=" * 70)

from src.rag.retriever import Retriever

retriever = Retriever()
print()

# 2. LLM
print("=" * 70)
print("🤖 [2/2] Carregando LLM (BioMistral + LoRA)...")
print("=" * 70)

from src.llm.client import LLMClient
llm = LLMClient(lora_path=Path("/content/biomistral-medquad-lora"))

print(f"\n⏱️  Tempo total: {time.time() - t0:.1f}s")

if llm.use_mock:
    print("\n⚠️  LLM EM MODO MOCK")
else:
    print("\n✅ LLM REAL carregado!")

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


📚 [1/2] Carregando RAG (ChromaDB)...


RuntimeError: Falha ao carregar modelo de embedding ('sentence-transformers/all-MiniLM-L6-v2'): numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

## Passo 10 - Subir UI

⚠️ **Importante**: Antes de subir a UI, garante que:
1. O patch do gradio_client foi aplicado (célula anterior)
2. O modelo foi copiado para `/content/Techchalleng3/biomistral-medquad-lora`
3. O RAG foi carregado (Passo 9)

Se aparecer `Internal Server Error`, é bug do gradio_client — patcha de novo.


In [ ]:
# Downgrade huggingface_hub pra versão que ainda tem HfFolder
!pip install -q "huggingface_hub==0.20.0"

# Verifica
import huggingface_hub
print(f"✅ huggingface_hub {huggingface_hub.__version__}")
print(f"✅ HfFolder disponível: {hasattr(huggingface_hub, 'HfFolder')}")

In [ ]:
import os
os.chdir("/content/Techchalleng3")

print("=" * 70)
print("🚀 SUBINDO UI GRADIO...")
print("=" * 70)
print("Aguarde 30s. Vai aparecer uma URL pública.")
print()

!python src/ui/gradio_app.py